# 🎌 YOLOv8 Segmentation Training — Comic Text Detection

**Notebook này hướng dẫn train YOLOv8 Segmentation model để phát hiện 3 loại vùng text trong truyện tranh:**
- `speech_bubble` — Bong bóng chat thông thường
- `sound_effect` — Âm thanh, hiệu ứng đánh nhau
- `calligraphy` — Thư pháp, chiêu thức, title chương

**YOLOv8 Segmentation** hỗ trợ **polygon** (bo nhiều góc), không chỉ hình chữ nhật!

---
**Yêu cầu:**
1. Dataset đã được export từ `web_labeler.py` (nút `📦 Export YOLO`)
2. Upload thư mục `yolo_export/` lên Google Drive
3. Chạy từng cell theo thứ tự

## 0️⃣ Kiểm Tra GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU sẵn sàng!')
    print(result.stdout)
else:
    print('⚠️  Không tìm thấy GPU!')
    print('→ Vào Runtime > Change runtime type > T4 GPU')

## 1️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive đã mount!')

## 2️⃣ Cài Đặt Thư Viện

In [ ]:
# Cài ultralytics (YOLOv8)
!pip install ultralytics -q

import ultralytics
print(f'✅ Ultralytics version: {ultralytics.__version__}')

from ultralytics import YOLO
import os, yaml, shutil
from pathlib import Path
print('✅ Import thành công!')

## 3️⃣ Cấu Hình Đường Dẫn Dataset

> **⚠️ Sửa cell này** để đúng với vị trí bạn upload dataset lên Drive!

In [ ]:
# ============================================================
# ⚙️ THAY ĐỔI CÁC ĐƯỜNG DẪN NÀY CHO ĐÚNG VỚI DRIVE CỦA BẠN
# ============================================================

# Đường dẫn thư mục yolo_export/ trên Google Drive
DATASET_PATH = '/content/drive/MyDrive/comic_dataset/yolo_export'

# Nơi lưu model sau khi train (trên Drive để không mất)
OUTPUT_PATH  = '/content/drive/MyDrive/comic_dataset/models'

# ============================================================

# Kiểm tra dataset tồn tại
data_yaml = os.path.join(DATASET_PATH, 'data.yaml')
if os.path.exists(data_yaml):
    print(f'✅ Dataset tìm thấy: {DATASET_PATH}')
    with open(data_yaml) as f:
        content = f.read()
    print('\n📄 Nội dung data.yaml:')
    print(content)
else:
    print(f'❌ Không tìm thấy data.yaml tại: {data_yaml}')
    print('→ Hãy kiểm tra lại đường dẫn DATASET_PATH!')

os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'📁 Output sẽ lưu tại: {OUTPUT_PATH}')

## 4️⃣ Cập Nhật `data.yaml` Với Đường Dẫn Colab

In [ ]:
# Đọc data.yaml và cập nhật path cho đúng với Colab
with open(data_yaml, 'r', encoding='utf-8') as f:
    data_config = yaml.safe_load(f)

# Cập nhật path tuyệt đối
data_config['path'] = DATASET_PATH
data_config['train'] = 'images/train'
data_config['val']   = 'images/train'  # Dùng tạm train set làm val nếu chưa có val riêng

# Lưu lại
colab_yaml = '/content/data.yaml'
with open(colab_yaml, 'w', encoding='utf-8') as f:
    yaml.dump(data_config, f, allow_unicode=True, default_flow_style=False)

print('✅ data.yaml đã cập nhật:')
with open(colab_yaml) as f:
    print(f.read())

# Đếm số ảnh
train_imgs = list(Path(DATASET_PATH, 'images', 'train').glob('*.*'))
train_lbls = list(Path(DATASET_PATH, 'labels', 'train').glob('*.txt'))
print(f'📊 Số ảnh train: {len(train_imgs)}')
print(f'📊 Số label:     {len(train_lbls)}')

# Kiểm tra format label (in 1 file mẫu)
if train_lbls:
    print('\n📄 Mẫu label file (polygon format):')
    with open(train_lbls[0]) as f:
        lines = f.readlines()
    for line in lines[:3]:
        parts = line.strip().split()
        class_id = int(parts[0])
        n_points = (len(parts) - 1) // 2
        print(f'  class={class_id}, polygon={n_points} điểm: {" ".join(parts[:5])}...')

## 5️⃣ Train YOLOv8 Segmentation

**Model size options:**
- `yolov8n-seg.pt` — Nano (nhỏ nhất, nhanh nhất, kém nhất)
- `yolov8s-seg.pt` — Small (**Khuyến nghị** cho truyện tranh)
- `yolov8m-seg.pt` — Medium (tốt hơn, cần nhiều VRAM hơn)
- `yolov8l-seg.pt` — Large (rất tốt, cần T4/A100)

> **💡 Gợi ý:** Với T4 GPU miễn phí, dùng `yolov8s-seg.pt`, epochs=100

In [ ]:
# ============================================================
# ⚙️ THAM SỐ TRAINING — Điều chỉnh theo nhu cầu
# ============================================================
MODEL_SIZE  = 'yolov8s-seg.pt'   # Khuyến nghị cho T4 GPU
EPOCHS      = 100                 # Số epoch (tăng lên 200 nếu dataset nhỏ)
BATCH_SIZE  = 8                   # Giảm xuống 4 nếu bị OOM
IMG_SIZE    = 1280                # Kích thước ảnh đầu vào (cao hơn = tốt hơn với truyện)
PROJECT     = OUTPUT_PATH         # Nơi lưu kết quả
RUN_NAME    = 'comic_seg_v1'      # Tên run
# ============================================================

print(f'🚀 Bắt đầu train với:')
print(f'   Model:      {MODEL_SIZE}')
print(f'   Epochs:     {EPOCHS}')
print(f'   Batch size: {BATCH_SIZE}')
print(f'   Img size:   {IMG_SIZE}')
print(f'   Output:     {PROJECT}/{RUN_NAME}\n')

# Load model pre-trained
model = YOLO(MODEL_SIZE)

# Bắt đầu train!
results = model.train(
    data      = colab_yaml,
    epochs    = EPOCHS,
    batch     = BATCH_SIZE,
    imgsz     = IMG_SIZE,
    project   = PROJECT,
    name      = RUN_NAME,
    device    = 0,           # GPU 0
    patience  = 30,          # Early stopping nếu không cải thiện sau 30 epochs
    save      = True,
    save_period = 20,        # Lưu checkpoint mỗi 20 epoch
    val       = True,
    plots     = True,
    # Augmentation tốt cho truyện tranh (B&W + màu)
    flipud    = 0.2,
    fliplr    = 0.5,
    degrees   = 10,
    translate = 0.1,
    scale     = 0.5,
    mosaic    = 1.0,
    # Cho ảnh truyện tranh thường là B&W
    hsv_h     = 0.01,
    hsv_s     = 0.3,
    hsv_v     = 0.4,
)

print('\n✅ Training hoàn tất!')
print(f'📊 Best mAP50: {results.results_dict.get("metrics/mAP50(M)", "N/A")}')

## 6️⃣ Xem Kết Quả Training

In [ ]:
from IPython.display import Image, display
import glob

run_dir = os.path.join(PROJECT, RUN_NAME)

# Hiển thị biểu đồ training
results_plot = os.path.join(run_dir, 'results.png')
if os.path.exists(results_plot):
    print('📊 Training curves:')
    display(Image(results_plot))

# Confusion matrix
conf_matrix = os.path.join(run_dir, 'confusion_matrix.png')
if os.path.exists(conf_matrix):
    print('\n📊 Confusion matrix:')
    display(Image(conf_matrix))

# Best model path
best_model = os.path.join(run_dir, 'weights', 'best.pt')
print(f'\n🏆 Best model: {best_model}')
if os.path.exists(best_model):
    size_mb = os.path.getsize(best_model) / 1024 / 1024
    print(f'   Kích thước: {size_mb:.1f} MB ✅')

## 7️⃣ Test Model Với Ảnh Mới

In [ ]:
import cv2
import numpy as np
from IPython.display import Image as IPImage, display

# ============================================================
# ⚙️ ĐỔI ĐƯỜNG DẪN ẢNH TEST
TEST_IMAGE = '/content/drive/MyDrive/comic_dataset/test/page_001.jpg'
# ============================================================

# Load best model
best_model_path = os.path.join(PROJECT, RUN_NAME, 'weights', 'best.pt')
model = YOLO(best_model_path)

# Màu cho từng class
CLASS_COLORS = {
    0: (232, 65, 24),    # speech_bubble — đỏ
    1: (230, 126, 34),   # sound_effect  — cam
    2: (142, 68, 173),   # calligraphy   — tím
}
CLASS_NAMES = ['speech_bubble', 'sound_effect', 'calligraphy']

if os.path.exists(TEST_IMAGE):
    results = model.predict(TEST_IMAGE, conf=0.3, imgsz=1280)

    img = cv2.imread(TEST_IMAGE)

    for r in results:
        # Vẽ mask polygon
        if r.masks is not None:
            for i, (mask, box) in enumerate(zip(r.masks.xy, r.boxes)):
                class_id = int(box.cls[0])
                color    = CLASS_COLORS.get(class_id, (255, 255, 255))
                conf     = float(box.conf[0])

                # Vẽ polygon
                pts = np.array(mask, dtype=np.int32)
                cv2.polylines(img, [pts], isClosed=True, color=color, thickness=3)

                # Overlay mờ
                overlay = img.copy()
                cv2.fillPoly(overlay, [pts], color)
                img = cv2.addWeighted(img, 0.85, overlay, 0.15, 0)

                # Label
                x1, y1 = int(box.xyxy[0][0]), int(box.xyxy[0][1])
                label = f"{CLASS_NAMES[class_id]} {conf:.2f}"
                cv2.rectangle(img, (x1, y1 - 22), (x1 + len(label)*8, y1), color, -1)
                cv2.putText(img, label, (x1 + 2, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
        else:
            # Detection model (không có mask)
            for box in r.boxes:
                x1,y1,x2,y2 = map(int, box.xyxy[0])
                class_id = int(box.cls[0])
                conf     = float(box.conf[0])
                color    = CLASS_COLORS.get(class_id, (255,255,255))
                cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)

    # Lưu và hiển thị
    out_path = '/content/test_result.jpg'
    cv2.imwrite(out_path, img)
    print(f'✅ Kết quả lưu tại: {out_path}')
    display(IPImage(out_path))
else:
    print(f'❌ Không tìm thấy ảnh test: {TEST_IMAGE}')
    print('→ Đổi đường dẫn TEST_IMAGE cho đúng')

## 8️⃣ Sao Chép Model Về Drive

In [ ]:
# Model đã được lưu tự động vào PROJECT/RUN_NAME/weights/
# Cell này copy thêm một bản vào thư mục dễ tìm hơn

best_model = os.path.join(PROJECT, RUN_NAME, 'weights', 'best.pt')
last_model = os.path.join(PROJECT, RUN_NAME, 'weights', 'last.pt')

final_dir = os.path.join(OUTPUT_PATH, 'final_models')
os.makedirs(final_dir, exist_ok=True)

if os.path.exists(best_model):
    dst = os.path.join(final_dir, 'yolov8_bubble_seg.pt')
    shutil.copy2(best_model, dst)
    print(f'✅ Best model đã copy → {dst}')
    print(f'   Kích thước: {os.path.getsize(dst)/1024/1024:.1f} MB')

print('\n📋 Hướng dẫn dùng model trên máy local:')
print('1. Download file yolov8_bubble_seg.pt từ Drive')
print(f'2. Đặt vào: d:\\PTUD\\mudule xu ly anh\\models\\yolov8_bubble_seg.pt')
print('3. Chạy main.py — pipeline sẽ tự động load model!')

## 9️⃣ (Nâng Cao) Tiếp Tục Train Từ Checkpoint

In [ ]:
# Nếu session Colab bị ngắt giữa chừng, dùng cell này để tiếp tục

# Đường dẫn checkpoint lần train trước
RESUME_FROM = os.path.join(PROJECT, RUN_NAME, 'weights', 'last.pt')

if os.path.exists(RESUME_FROM):
    model = YOLO(RESUME_FROM)
    results = model.train(
        resume = True,   # Tiếp tục từ checkpoint!
        device = 0,
    )
    print('✅ Resume training thành công!')
else:
    print(f'❌ Không tìm thấy checkpoint: {RESUME_FROM}')

## 🔟 (Tùy Chọn) Chia Train/Val Dataset

Nếu bạn có nhiều ảnh (> 100), nên tách riêng tập val để đánh giá chính xác hơn.

In [ ]:
import random

# ============================================================
VAL_RATIO = 0.15   # 15% làm validation set
# ============================================================

train_img_dir = os.path.join(DATASET_PATH, 'images', 'train')
val_img_dir   = os.path.join(DATASET_PATH, 'images', 'val')
train_lbl_dir = os.path.join(DATASET_PATH, 'labels', 'train')
val_lbl_dir   = os.path.join(DATASET_PATH, 'labels', 'val')

os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(val_lbl_dir, exist_ok=True)

all_imgs = list(Path(train_img_dir).glob('*.*'))
random.shuffle(all_imgs)
val_count = max(1, int(len(all_imgs) * VAL_RATIO))
val_imgs  = all_imgs[:val_count]

for img_path in val_imgs:
    # Move ảnh
    shutil.move(str(img_path), val_img_dir)
    # Move label
    lbl_path = Path(train_lbl_dir) / (img_path.stem + '.txt')
    if lbl_path.exists():
        shutil.move(str(lbl_path), val_lbl_dir)

# Cập nhật data.yaml với val path
data_config['val'] = 'images/val'
with open(colab_yaml, 'w') as f:
    yaml.dump(data_config, f, allow_unicode=True, default_flow_style=False)

remaining = list(Path(train_img_dir).glob('*.*'))
print(f'✅ Đã tách dataset:')
print(f'   Train: {len(remaining)} ảnh')
print(f'   Val:   {len(val_imgs)} ảnh')
print('\n→ Bây giờ chạy lại Cell 5 để train với val set chính thức!')